In [2]:
import jax 
import jax.numpy as jnp

from probjax.nn.attention import dense_dot_product_attention, efficient_dot_product_attention, efficient_masked_dot_product_attention


In [3]:
with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True):
  # Run the operations to be profiled
  key = jax.random.PRNGKey(0)
  x = jax.random.normal(key, (5000, 5000))
  y = x @ x
  y.block_until_ready()

I0000 00:00:1702996602.800750  297579 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.
2023-12-19 15:36:45.407179: W external/xla/xla/service/gpu/nvptx_compiler.cc:708] The NVIDIA driver's CUDA version is 11.7 which is older than the ptxas CUDA version (11.8.89). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


Open URL in browser: https://ui.perfetto.dev/#!/?url=http://127.0.0.1:9001/perfetto_trace.json.gz


KeyboardInterrupt: 

In [4]:
def efficient_masked_dot_product_attention(
    query_heads,  # [...,T', H, K]
    key_heads,  # [...,T', H, K]
    value_heads,  # [T, H, V]
    indices1,  # Should be the indices where the mask is true
    indices2,
    return_attention_weights: bool = False,
):
    *leading_dims, sequence_length, _, dim = query_heads.shape
    query_heads = jnp.take(
        query_heads, indices1, axis=-3
    )  # [..., E, H, K] Where E is the number of edges
    key_heads = jnp.take(key_heads, indices2, axis=-3)  # [..., E, H, K]
    value_heads = jnp.take(value_heads, indices2, axis=-3)  # [..., E, H, V]

    # Attention logits
    attention_logits = jnp.einsum(
        "...ehd,...ehd->...eh", query_heads, key_heads
    ) / jnp.sqrt(dim).astype(key_heads.dtype)
    attention_logits = attention_logits - jnp.max(
        attention_logits, axis=-2, keepdims=True
    )
    attention_weight = jnp.exp(attention_logits)
    attention_normalizer = jax.ops.segment_sum(
        attention_weight,
        indices1,
        num_segments=sequence_length,
        indices_are_sorted=True,
    )
    attention_normalizer = jnp.take(attention_normalizer, indices1, axis=-2)
    attention_weight = attention_weight / attention_normalizer  # [..., eh]

    # Attention weighted values
    attn = attention_weight[..., None] * value_heads
    attn = jax.ops.segment_sum(
        attn, indices1, num_segments=sequence_length, indices_are_sorted=True
    )
    attn = jnp.reshape(attn, (*leading_dims, sequence_length, -1))  # [T', H*V]

    if return_attention_weights:
        return attn, attention_weight
    else:
        return attn, None

In [42]:

mask = jax.random.bernoulli(jax.random.PRNGKey(0), p=0.5, shape=(1,1, 5000, 5000)).astype(bool)

q = jax.random.normal(jax.random.PRNGKey(0), shape=(1, 5000,4, 10))
k = jax.random.normal(jax.random.PRNGKey(1), shape=(1, 5000,4, 10))
v = jax.random.normal(jax.random.PRNGKey(2), shape=(1, 5000,4, 10))

In [50]:
@jax.jit
def f1():
    return dense_dot_product_attention(q, k, v, mask=mask, key_size=2)[0]


In [58]:
@jax.jit
def f2():
    return efficient_dot_product_attention(q, k, v, mask=mask)

In [53]:
%%timeit
f1()

5.44 ms ± 97.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [55]:
%%timeit
f2()

4.56 ms ± 37.7 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [57]:
jnp.allclose(f1(), f2())

ValueError: Incompatible shapes for broadcasting: shapes=[(1, 5000, 40), (20476, 10)]

In [12]:
jax.profiler.stop_trace()

OSError: [Errno 98] Address already in use

In [9]:
with jax.profiler.trace("/tmp/jax-trace", create_perfetto_link=True, create_perfetto_trace=True):
  # Run the operations to be profiled
  y = f()
  y.block_until_ready()

RuntimeError: Profile has already been started. Only one profile may be run at a time.